<a href="https://colab.research.google.com/github/louweiqun/dino-quiz-adventure/blob/main/ComfyUI_Video_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ComfyUI 视频生成环境（Google Colab 一键部署）**默认方案：Wan2.2 5B TI2V** —— 一套模型同时支持「文生视频」和「图生视频」，免费 T4 显卡可跑。### 使用前必做1. 菜单 **运行时 → 更改运行时类型 → 硬件加速器选 T4 GPU**（A100/L4 更快）。2. 从上到下依次运行每个代码单元。3. 运行完第 ⑧ 单元后，**在单元格输出里找「链接 A」**，复制到一个新标签页打开就是 ComfyUI 界面。4. 如果看不到链接，直接运行第 ⑧b 诊断单元；也可以看 Colab 左侧「外部服务」面板里的 8188 端口。> 模型首次下载约 **17 GB**，请保持页面打开。隧道链接没有密码，不要公开分享。

In [4]:
import torch, subprocess
print('=== HARDWARE CHECK ===')
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0)
    print('GPU:', p.name, '| VRAM: %.1f GB' % (p.total_memory/1024**3))
else:
    print('NO GPU')
r=subprocess.run(['bash','-lc','free -g | head -2; echo ---DISK---; df -h /content | tail -1'], capture_output=True, text=True)
print(r.stdout)
print('=== END CHECK ===')

=== HARDWARE CHECK ===
GPU: Tesla T4 | VRAM: 14.6 GB
               total        used        free      shared  buff/cache   available
Mem:              12           1           8           0           3          11
---DISK---
overlay         113G   83M  113G   1% /

=== END CHECK ===


In [6]:
# ② 配置（按需修改）
import torch
PERSIST_TO_DRIVE = False  # True=存Drive跨会话复用; False=存Colab本地
LOW_VRAM = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory/1024**3 <= 16
print('PERSIST_TO_DRIVE =', PERSIST_TO_DRIVE)
print('LOW_VRAM =', LOW_VRAM)

PERSIST_TO_DRIVE = False
LOW_VRAM = True


In [7]:
# ③ 下载 / 更新 ComfyUI 本体
import os, subprocess
ROOT = '/content/ComfyUI'
if not os.path.exists(os.path.join(ROOT, 'main.py')):
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git', ROOT], check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull','--ff-only','-q'])
print('ComfyUI 就绪:', os.path.exists(os.path.join(ROOT,'main.py')))

ComfyUI 就绪: True


In [ ]:
# ④ 可选：把模型/输出目录指向 Google Drive（仅当 PERSIST_TO_DRIVE = True）
import os, shutil
DRIVE = '/content/drive/MyDrive/ComfyUI_Video'
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    subs = ['input','output','models/diffusion_models','models/text_encoders
models/vae','models/loras','models/checkpoints']
    for sub in subs:
        target = os.path.join(DRIVE, sub)
        local = os.path.join('/content/ComfyUI', sub)
        os.makedirs(target, exist_ok=True)
        if os.path.islink(local): continue
        if os.path.isdir(local):
            for f in os.listdir(local):
                src, dst = os.path.join(local, f), os.path.join(target, f)
                if not os.path.exists(dst): shutil.move(src, dst)
            shutil.rmtree(local)
        os.makedirs(os.path.dirname(local), exist_ok=True)
        os.symlink(target, local)
    print('已软链接到 Google Drive:', DRIVE)
else:
    print('使用 Colab 本地磁盘；断开连接后模型需要重新下载。')

In [8]:
# ⑤ 安装依赖
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','-U','pip'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/ComfyUI/requirements.txt'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub','hf_transfer','imageio[ffmpeg]','av'], check=False)
print('依赖安装完成')

依赖安装完成


In [9]:
# ⑥ 下载 Wan2.2 5B 模型（约 17GB，首次较慢）
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download
CACHE = os.path.join(DRIVE,'_hf_cache') if PERSIST_TO_DRIVE else '/content/_hf_cache'
MODELS = [
    ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors','diffusion_models'),
    ('Comfy-Org/Wan_2.1_ComfyUI_repackaged','split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors','text_encoders'),
    ('Comfy-Org/Wan_2.2_ComfyUI_Repackaged','split_files/vae/wan2.2_vae.safetensors','vae'),
]
for repo, filename, folder in MODELS:
    print("下载中:", filename, flush=True)
    src = hf_hub_download(repo_id=repo, filename=filename, cache_dir=CACHE)
    dst_dir = os.path.join('/content/ComfyUI/models', folder)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(filename))
    if os.path.lexists(dst): os.unlink(dst)
    os.symlink(src, dst)
    print("完成:", dst, round(os.path.getsize(src)/1024**3,2), "GB", flush=True)

下载中: split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


split_files/diffusion_models/wan2.2_ti2v(…): reconstructing file:   0%|          |  0.00B / 10.0GB            

split_files/diffusion_models/wan2.2_ti2v(…): downloading bytes:           |  0.00B            

完成: /content/ComfyUI/models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors 9.31 GB
下载中: split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors


split_files/text_encoders/umt5_xxl_fp8_e(…): reconstructing file:   0%|          |  0.00B / 6.74GB            

split_files/text_encoders/umt5_xxl_fp8_e(…): downloading bytes:           |  0.00B            

完成: /content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors 6.27 GB
下载中: split_files/vae/wan2.2_vae.safetensors


split_files/vae/wan2.2_vae.safetensors: reconstructing file:   0%|          |  0.00B / 1.41GB            

split_files/vae/wan2.2_vae.safetensors: downloading bytes:           |  0.00B            

完成: /content/ComfyUI/models/vae/wan2.2_vae.safetensors 1.31 GB


In [11]:
# ⑦ 下载官方工作流模板，并预设为 T4 友好参数
import os, json, urllib.request
WF_DIR = '/content/ComfyUI/user/default/workflows'
os.makedirs(WF_DIR, exist_ok=True)
wf_path = os.path.join(WF_DIR, 'Wan2.2_5B_TI2V.json')
url = 'https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_wan2_2_5B_ti2v.json'
urllib.request.urlretrieve(url, wf_path)
with open(wf_path, encoding="utf-8") as f: wf = json.load(f)
for node in wf.get("nodes", []):
    if node.get("type") == "Wan22ImageToVideoLatent":
        node["widgets_values"] = [832, 480, 81, 1]
    if node.get("type") == "CreateVideo":
        node["widgets_values"] = [16]
with open(wf_path, "w", encoding="utf-8") as f: json.dump(wf, f, ensure_ascii=False, indent=1)
print('工作流已放入 ComfyUI:', wf_path)

工作流已放入 ComfyUI: /content/ComfyUI/user/default/workflows/Wan2.2_5B_TI2V.json


In [12]:
# ⑧ 启动 ComfyUI 并打印访问链接（主线程输出，一定能看到）
import os, subprocess, threading, time, socket, urllib.request
PORT = 8188
log = open('/content/comfyui.log','w')
cmd = ['python','/content/ComfyUI/main.py','--listen','0.0.0.0','--port',str(PORT)]
if LOW_VRAM: cmd.append("--lowvram")
proc = subprocess.Popen(cmd, cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT)
ok = False
for _ in range(180):
    s = socket.socket()
    ok = s.connect_ex(('127.0.0.1',PORT)) == 0
    s.close()
    if ok: break
    time.sleep(1)
print('='*62)
print('ComfyUI 服务:', '已启动' if ok else '未启动（看下方日志）')
print('='*62)
try:
    from google.colab.kernel import proxyPort
    url = proxyPort(PORT)
    print('【链接 A · Colab 自带转发，推荐】')
    print(url)
except Exception as e:
    print('方式 A 不可用:', e)
print('【链接 B】Colab 左侧边栏 -> 外部服务 -> 点 8188 端口')
def try_cloudflared():
    try:
        bin_path = '/usr/local/bin/cloudflared'
        if not os.path.exists(bin_path):
            urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64","/tmp/cloudflared")
            subprocess.run(['chmod','+x','/tmp/cloudflared'], check=True)
            subprocess.run(['mv','/tmp/cloudflared',bin_path], check=True)
        t = subprocess.Popen([bin_path,'tunnel','--url','http://127.0.0.1:%d'%PORT], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in t.stdout:
            if 'trycloudflare.com' in line and 'https://' in line:
                link = line[line.find('https://'):].split()[0].rstrip('.,|')
                print('【链接 C · 备用公网链接】'+link, flush=True)
                break
    except Exception as e:
        print('方式 C 不可用（不影响 A/B）:', str(e)[:200], flush=True)
threading.Thread(target=try_cloudflared, daemon=True).start()
r = subprocess.run(['tail','-15','/content/comfyui.log'], capture_output=True, text=True)
print('日志最后 15 行：'+(r.stdout or '（空）'))

ComfyUI 服务: 已启动
方式 A 不可用: No module named 'google.colab.kernel'
【链接 B】Colab 左侧边栏 -> 外部服务 -> 点 8188 端口
日志最后 15 行：[INFO] comfyui-embedded-docs version: 0.5.17
[INFO] comfy-aimdo version: 0.6.0
[INFO] comfy-kitchen version: 0.2.42
[INFO] [Prompt Server] web root: /usr/local/lib/python3.13/dist-packages/comfyui_frontend_package/static
[INFO] Asset seeder disabled
[INFO] No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'
[INFO] 
Import times for custom nodes:
[INFO]    0.0 seconds: /content/ComfyUI/custom_nodes/websocket_image_save.py
[INFO] 
[INFO] Using RAM pressure cache.
[INFO] ComfyUI is developed by Comfy Org. We also offer commercial products to deploy ComfyUI as scalable cloud endpoints. Visit platform.comfy.org/ascii for more information.
[INFO] Starting server

[INFO] To see the GUI go to: http://0.0.0.0:8188



In [19]:
# 查询 H3 R2V 模型仓库文件（不影响已启动的隧道进程）
from huggingface_hub import HfApi
api = HfApi()
repo = 'Comfy-Org/MiniMax-H3'
print('REPO:', repo)
try:
    info = api.model_info(repo, files_metadata=True)
    for s in info.siblings:
        if s.rfilename.endswith('.safetensors'):
            print('FILE:', s.rfilename, round((s.size or 0)/1024**3, 2), 'GB')
except Exception as e:
    print('ERR:', str(e)[:300])

REPO: Comfy-Org/MiniMax-H3
FILE: diffusion_models/minimax_h3_fl2va_bf16.safetensors 61.73 GB
FILE: diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors 31.7 GB
FILE: diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors 37.46 GB
FILE: diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors 19.52 GB
FILE: diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors 19.53 GB
FILE: diffusion_models/minimax_h3_fl2va_pruned_w6a8.safetensors 14.89 GB
FILE: diffusion_models/minimax_h3_ref2va_bf16.safetensors 61.73 GB
FILE: diffusion_models/minimax_h3_ref2va_int8_convrot.safetensors 31.7 GB
FILE: diffusion_models/minimax_h3_ref2va_pruned_bf16.safetensors 37.46 GB
FILE: diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors 19.52 GB
FILE: diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors 19.53 GB
FILE: diffusion_models/minimax_h3_ref2va_pruned_w6a8.safetensors 14.89 GB
FILE: embeddings/minimaxh3_art_is_explosion.safetensors 0.0 GB
FILE: embeddin

## 怎么出片1. 打开上面打印的链接，进入 ComfyUI。2. 菜单 **Workflow → Open Workflow**，选择 `Wan2.2_5B_TI2V.json`。3. **文生视频**：直接改 Positive Prompt / Negative Prompt 两个节点的文字。4. **图生视频**：选中左下角 `Load Image` 节点，按 **Ctrl+B** 取消禁用（默认是灰的），上传你的图片。5. **时长 / 尺寸**：`Wan22ImageToVideoLatent` 节点，当前 832x480、81 帧（16fps ≈ 5 秒）。T4 别超过 832x480 / 81 帧；A100 可以到 1280x704 / 121 帧。6. 点右侧 **Run**（或 Ctrl + Enter）开始生成，T4 大约几分钟一条。7. 成片在 `ComfyUI/output/video/`，也可以运行下面这个单元直接预览。> 提示词建议按「主体 + 动作 + 场景 + 镜头 + 光线 + 风格」写，一条 5 秒视频只描述一个动作，画面会更稳。

In [ ]:
# ⑨ 预览最近生成的视频（可选）
import os, glob
from IPython.display import Video, display
files = sorted(glob.glob('/content/ComfyUI/output/video/*.*'), key=os.path.getmtime, reverse=True)
print('已生成视频数:', len(files))
for f in files[:5]: print(f)
if files: display(Video(files[0], width=640))

## 常见问题| 现象 | 处理 ||---|---|| 第 ⑧ 单元没打印链接 | 运行第 ⑧b 诊断单元，它会显示状态和 Colab 自带链接 || 打不开链接 | 优先用「链接 A」；或 Colab 左侧「外部服务」→ 8188 端口 || 诊断显示端口没监听 | 前面单元格有红色报错（多半是模型没下完），或重跑第 ⑧ 单元 || CUDA out of memory | 帧数降到 49 或 33，分辨率降到 640x384，运行时 → 重启会话 || 每次重连都要下模型 | 第 ② 单元 `PERSIST_TO_DRIVE` 改成 `True`，模型缓存到 Google Drive || 提示缺少节点 | 重跑第 ③ 单元更新 ComfyUI || 想装 LoRA / 第三方节点 | 安装 ComfyUI-Manager： |```pythonimport subprocesssubprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ltdrdata/ComfyUI-Manager.git',                '/content/ComfyUI/custom_nodes/ComfyUI-Manager'])```